# Load Data

In [ ]:
import numpy as np
import pandas as pd
import pickle
import sys
from pathlib import Path

from sklearn.linear_model import RidgeCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler


# part_d.py — Config

In [ ]:
# %%writefile part_d.py
FORMAT_VERSION = 1


# Feature engineering\n\nSingle source of truth for turning one raw `.npz` window into a feature row. Used identically by both `train` and `feature_engineering` modes.

In [ ]:
# %%writefile -a part_d.py

def _stats(x, prefix, names):
    """
    Basic summary stats for a 1D array, ignoring NaNs. Returns values + appends names.
    If a window has no valid samples at all for this modality, emits np.nan rather
    than a hardcoded fallback (e.g. 0.0) — a fabricated constant would silently
    distort downstream model fitting/interpretation, whereas np.nan is handled
    explicitly and only ever imputed using train-set statistics (see cmd_train).
    """
    x = x[~np.isnan(x)]
    if x.size == 0:
        vals = [np.nan, np.nan, np.nan, np.nan, np.nan]
    else:
        vals = [
            float(np.mean(x)),
            float(np.std(x)),
            float(np.min(x)),
            float(np.max(x)),
            float(np.max(x) - np.min(x)),
        ]
    names.extend([f"{prefix}_mean", f"{prefix}_std", f"{prefix}_min", f"{prefix}_max", f"{prefix}_range"])
    return vals


def _slope(x, prefix, names):
    """Least-squares slope of a 1D array against sample index, ignoring NaNs."""
    idx = np.arange(x.shape[0])
    mask = ~np.isnan(x)
    if mask.sum() < 2:
        val = np.nan
    else:
        val = float(np.polyfit(idx[mask], x[mask], 1)[0])
    names.append(f"{prefix}_slope")
    return [val]


def make_features_for_window(record, feature_names_out):
    """
    record: dict-like with per-window 1D/2D arrays already sliced for ONE example,
            e.g. record["e4_bvp"] has shape (19200,), record["e4_acc"] has shape (9600, 3).
    feature_names_out: list to append feature names to (only populated on first call;
                        caller is responsible for only using this on the first row and
                        asserting consistency afterwards).
    Returns: 1D numpy array of feature values for this example.
    """
    names = []
    vals = []

    # --- BVP (E4 PPG) ---
    bvp = record["e4_bvp"]
    vals += _stats(bvp, "bvp", names)
    vals += _slope(bvp, "bvp", names)

    # --- E4 HR (device-derived heart rate trace) ---
    hr = record["e4_hr"]
    vals += _stats(hr, "e4_hr", names)

    # --- EDA ---
    eda = record["e4_eda"]
    vals += _stats(eda, "eda", names)
    vals += _slope(eda, "eda", names)

    # --- Temperature ---
    temp = record["e4_temp"]
    vals += _stats(temp, "temp", names)
    vals += _slope(temp, "temp", names)

    # --- E4 accelerometer: squared magnitude a_sq(t) = ax^2+ay^2+az^2 ---
    acc = record["e4_acc"]  # (T, 3)
    acc_sq = np.nansum(acc.astype(np.float64) ** 2, axis=1)
    vals += _stats(acc_sq, "e4_acc_sq", names)

    # --- Zephyr ECG ---
    ecg = record["zephyr_ecg"]
    vals += _stats(ecg, "ecg", names)

    # --- Zephyr accelerometer magnitude ---
    zacc = record["zephyr_acc"]
    zacc_sq = np.nansum(zacc.astype(np.float64) ** 2, axis=1)
    vals += _stats(zacc_sq, "zephyr_acc_sq", names)

    # --- Zephyr breathing ---
    breathing = record["zephyr_breathing"]
    vals += _stats(breathing, "breathing", names)

    if not feature_names_out:
        feature_names_out.extend(names)
    else:
        assert feature_names_out == names, "Feature name/order mismatch between windows"

    return np.array(vals, dtype=np.float64)


# Streaming file processing\n\nProcesses one `.npz` file at a time to keep memory bounded (indexing into an npz array still loads the full array first, so we must avoid holding many files in memory at once).

In [ ]:
# %%writefile -a part_d.py

def iter_participant_files(data_dir):
    return sorted(Path(data_dir).glob("*.npz"))


def build_feature_matrix(data_dir, has_target):
    """
    Processes one .npz file at a time to keep memory bounded.
    Returns (Z, y, feature_names) where y is None if has_target is False.
    """
    feature_names = []
    feature_rows = []
    targets = [] if has_target else None

    for path in iter_participant_files(data_dir):
        with np.load(path, allow_pickle=False) as data:
            n = data["e4_bvp"].shape[0]

            fields = {
                "e4_bvp": data["e4_bvp"],
                "e4_hr": data["e4_hr"],
                "e4_eda": data["e4_eda"],
                "e4_temp": data["e4_temp"],
                "e4_acc": data["e4_acc"],
                "zephyr_ecg": data["zephyr_ecg"],
                "zephyr_acc": data["zephyr_acc"],
                "zephyr_breathing": data["zephyr_breathing"],
            }

            if has_target:
                glucose = data["glucose"]

            for i in range(n):
                record = {k: v[i] for k, v in fields.items()}
                row = make_features_for_window(record, feature_names)
                feature_rows.append(row)

            if has_target:
                targets.append(glucose)

    Z = np.vstack(feature_rows)
    y = np.concatenate(targets) if has_target else None
    return Z, y, feature_names


# Train / feature_engineering entry points

In [ ]:
# %%writefile -a part_d.py

def cmd_train(protocol, train_dir, model_path):
    Z, y, feature_names = build_feature_matrix(train_dir, has_target=True)

    # Z may contain NaN columns for windows where a modality had no valid
    # samples at all (see _stats/_slope). Impute with the per-feature median,
    # fit using training data only, per the assignment's leakage rules.
    imputer = SimpleImputer(strategy="median")
    Z_imputed = imputer.fit_transform(Z)

    scaler = StandardScaler()
    Z_scaled = scaler.fit_transform(Z_imputed)

    model = RidgeCV(alphas=np.logspace(-3, 3, 13))
    model.fit(Z_scaled, y)

    # Convert standardized-space coefficients back to raw feature scale so that
    # saved (intercept, coef) act directly on the *unscaled* engineered features.
    # y_hat = intercept_s + coef_s . ((z - mean) / scale)
    #       = (intercept_s - sum(coef_s * mean / scale)) + sum((coef_s/scale) * z)
    coef_scaled = model.coef_
    coef_raw = coef_scaled / scaler.scale_
    intercept_raw = model.intercept_ - np.sum(coef_scaled * scaler.mean_ / scaler.scale_)

    state = {
        "format_version": FORMAT_VERSION,
        "protocol": protocol,
        "intercept": float(intercept_raw),
        "coef": coef_raw.astype(np.float64),
        "feature_names": feature_names,
        "preprocessing_state": {
            "imputer": imputer,
        },
    }

    with open(model_path, "wb") as f:
        pickle.dump(state, f)

    print(f"[train] protocol={protocol} n={Z.shape[0]} m={Z.shape[1]} "
          f"best_alpha={model.alpha_:.4g}")


def cmd_feature_engineering(protocol, test_dir, model_path, output_path):
    with open(model_path, "rb") as f:
        state = pickle.load(f)

    assert state["protocol"] == protocol, "Model/protocol mismatch"

    Z, _, feature_names = build_feature_matrix(test_dir, has_target=False)
    assert feature_names == state["feature_names"], "Feature name/order mismatch vs. training"

    imputer = state["preprocessing_state"]["imputer"]
    Z_imputed = imputer.transform(Z)

    assert np.isfinite(Z_imputed).all(), "Non-finite values remain in final feature matrix"

    np.save(output_path, Z_imputed)
    print(f"[feature_engineering] protocol={protocol} n={Z_imputed.shape[0]} m={Z_imputed.shape[1]}")


# CLI

In [ ]:
# %%writefile -a part_d.py

def main():
    if len(sys.argv) < 2:
        print("usage:\n"
              "  part_d.py train <protocol> <train_dir> <model_path>\n"
              "  part_d.py feature_engineering <protocol> <test_dir> <model_path> <output_path>")
        sys.exit(1)

    mode = sys.argv[1]

    if mode == "train":
        _, _, protocol, train_dir, model_path = sys.argv
        cmd_train(protocol, train_dir, model_path)
    elif mode == "feature_engineering":
        _, _, protocol, test_dir, model_path, output_path = sys.argv
        cmd_feature_engineering(protocol, test_dir, model_path, output_path)
    else:
        raise ValueError(f"Unknown mode: {mode}")


if __name__ == "__main__":
    main()


# For testing on Kaggle

In [ ]:
base_dir = "/kaggle/input/datasets/souravkumarpatel/assignment1-cgm-glucose-estimation/"

# !python3 part_d.py train d1 {base_dir}train_d1/ model_d1.pkl
# !python3 part_d.py feature_engineering d1 {base_dir}test_d1/ model_d1.pkl features_d1.npy
